In [1]:
%pip install backtesting

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 175.5/175.5 kB 2.7 MB/s eta 0:00:00a 0:00:01
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.8/6.8 MB 6.0 MB/s eta 0:00:0000:0100:01m
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 kB 2.5 MB/s eta 0:00:00
  Created wheel for backtesting: filename=Backtesting-0.3.3-py3-none-any.whl size=173916 sha256=1c248988e71666d2a7fc6b9a84ec330772a52a3389cc4e01f6f4fc0293931760
  Stored in directory: /Users/parthajit/Library/Caches/pip/wheels/2c/56/19/bf7ee5e164aa99a976e3f64841c83b5ae0391c59d9aec011d0
Successfully built backtesting
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip is available: 23.1.2 -> 23.3.1
[notice] To update, run: python3 -m pip install --upgrade pip
Note:

In [6]:
# implement a backtesting engine that takes in 1 csv containing columns datetime,open,high,low,close,volume 
# and another csv containing the same columns but with the strategy's buy/sell signals added and returns
# multiple performance metrics and the graph

'''
Static
performance metrics to calculate:-
1. total trades
2. winning trades
3. losing trades
4. benchmark return (buy and hold)
5. win rate
6. gross profit
7. net profit
8. average profit 
9. maxiumum drawdown (percentage)
10. largest win
11. average win
12. largest loss
13. average loss
14. max holding period
15. average holding period
16. max dip
17. average dip
18. sharpe ratio
19. sortino ratio
20. from and to dates

Compounding
performance metrics to calculate:-
1. Initial Balance
2. Number of trades
3. Max PNL
4. Min PNL
5. Peak Portfolio Balance
6. Lowest Portfolio Balance
7. End Portfolio Balance
8. Total Fee
'''



In [11]:
#import both csvs:- 1. historical data 2. strategy signals
import backtesting
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import datetime
import math
from datetime import datetime

#read csvs
#historical data
hist = pd.read_csv('/Users/parthajit/Desktop/InterIIT/ZeltaLab_InterIIT/InterIIT-CryptoTrading/btcusdt_1h.csv')
strat = pd.read_csv('/Users/parthajit/Desktop/InterIIT/ZeltaLab_InterIIT/InterIIT-CryptoTrading/signals.csv')

#format of hist
#datetime,open,high,low,close,volume

#format of strat
#datetime,signals,open,high,low,close,volume,


# 1. open long
class backtest():
    def __init__(self, hist, strat):
        self.is_static = True
        self.is_long = False
        self.hist = hist
        self.strat = strat
        self.signals = self.extract_signals()
        self.capital = 1000 # fixed capital in static   #constant        
        self.total_portfolio_static = 1000
        self.pnl = []
        self.total_portfolio_compound = 1000
        self.static_balance = []
        self.compound_balance = []
        #self.hist['compound_balance'] = self.compound_balance
        self.total_pnl = 0 #compounding
        self.number_of_btc=0
        self.fee = 0.0015
        self.trades = 0
        self.win_trades = 0
        self.lose_trades = 0
        self.benchmark = 0
        self.win_rate = 0
        self.gross_profit = 0
        self.net_profit = 0
        self.avg_profit = 0
        self.max_drawdown = 0
        self.largest_win = 0
        self.avg_win = 0
        self.largest_loss = 0
        self.avg_loss = 0
        self.max_holding_period = 0
        self.avg_holding_period = 0
        self.max_dip = 0
        self.avg_dip = 0
        self.sharpe_ratio = 0
        self.sortino_ratio = 0
        self.is_closed = True
        self.initial_price = 0
        self.final_price = 0
        #extract to and from datetime
        self.from_datetime = self.hist['datetime'][0]
        self.to_datetime = self.hist['datetime'][len(self.hist)-1]
        self.entry_times = []
        self.exit_times = []

    # 5 cases:-
    # 1. open long
    # 2. close long
    # 3. open short
    # 4. close short
    # 5. do nothing

    def extract_signals(self):
        #extract signals from strat
        signals = self.strat['signals']
        return signals
    
    def transaction_cost(self, price):
        return self.fee*price
    
    def open_long(self, index):
        #buy at opening price
        self.initial_price = self.hist['open'][index]
        self.is_closed = False
        self.trades += 1
        self.is_long = True
        self.entry_times.append(self.hist['datetime'][index])
        #static
        if self.is_static:
            self.total_portfolio_static -= self.transaction_cost(self.capital)
            self.total_portfolio_static -= self.capital
            self.number_of_btc += self.capital/self.hist['open'][index]
            #calculate transaction cost
        #compounding
        else:
            self.number_of_btc += self.total_portfolio_compound/self.hist['open'][index]
            curr_transaction_cost = self.transaction_cost(self.total_portfolio_compound)
            self.total_portfolio_compound = 0
            self.total_portfolio_compound -= curr_transaction_cost
            #calculate transaction cost
    

    def open_short(self, index):
        #sell at closing price
        self.initial_price = self.hist['close'][index]
        self.is_closed = False
        self.trades += 1
        self.entry_times.append(self.hist['datetime'][index])
        self.is_long = False
        #static
        if self.is_static:
            self.number_of_btc -= self.capital/self.hist['close'][index]
            self.total_portfolio_static += self.capital
            self.total_portfolio_static -= self.transaction_cost(self.capital)
        #compounding
        else:
            self.number_of_btc -= self.total_portfolio_compound/self.hist['close'][index]
            self.total_portfolio_compound -= self.transaction_cost(self.total_portfolio_compound)
            self.total_portfolio_compound += (-self.number_of_btc)*self.hist['close'][index]
            


    def close_long(self, index):
        #sell at closing price
        self.final_price = self.hist['close'][index]
        self.is_closed = True
        current_pnl = self.number_of_btc*(self.final_price-self.initial_price)
        self.total_pnl += current_pnl
        self.pnl.append(current_pnl)
        self.is_long = False
        self.exit_times.append(self.hist['datetime'][index])
        #static
        if self.is_static:
            self.total_portfolio_static += self.number_of_btc*self.hist['close'][index]
            self.static_balance.append(self.total_portfolio_static)
            self.number_of_btc = 0
        #compounding
        else:
            self.total_portfolio_compound += self.number_of_btc*self.hist['close'][index]
            self.compound_balance.append(self.total_portfolio_compound) 
            self.number_of_btc = 0


    def close_short(self, index):
        #buy at open      
        self.final_price = self.hist['open'][index]
        self.is_closed = True
        current_pnl = (-self.number_of_btc)*(self.initial_price-self.final_price)
        self.total_pnl += current_pnl
        self.pnl.append(current_pnl)
        self.is_long = False
        self.exit_times.append(self.hist['datetime'][index])
        #static
        if self.is_static:
            self.total_portfolio_static += (-self.number_of_btc)*self.hist['open'][index]
            self.static_balance.append(self.total_portfolio_static)
            self.number_of_btc = 0
        #compounding
        else:
            self.total_portfolio_compound += (-self.number_of_btc)*self.hist['open'][index]
            self.compound_balance.append(self.total_portfolio_compound)
            self.number_of_btc = 0
            

    def do_nothing(self, index): #evil floating point bit hack
        pass
    
    #plot graph of compound balance vs time 
    def plot_graph(self):
        # x axis is datetime
        # y axis is compound balance
        # plot compound balance vs time
        plt.plot(self.hist['datetime'], self.compound_balance)
        plt.xlabel('time')
        plt.ylabel('compound balance')
        plt.show()
   
    def calculate_gross_profit(self):
        self.gross_profit = sum(pnl for pnl in self.pnl if pnl > 0)

    def calculate_max_drawdown(self):
    # Convert self.static_balance to a pandas Series
        static_balance_series = pd.Series(self.static_balance)

        max_balance = static_balance_series.cummax()
        drawdown = ((static_balance_series - max_balance) / max_balance) * 100
        max_drawdown = max(drawdown)
        self.max_drawdown = max_drawdown

    def calculate_holding_periods(self):
    # Assuming you have a list of timestamps representing entry and exit times
        entry_times = [datetime.strptime(time_str, '%Y-%m-%d %H:%M:%S') for time_str in self.entry_times]
        exit_times = [datetime.strptime(time_str, '%Y-%m-%d %H:%M:%S') for time_str in self.exit_times]

    # Calculate the holding period for each trade
        holding_periods = [(exit_time - entry_time).total_seconds() / 3600 for entry_time, exit_time in zip(entry_times, exit_times)]

        if holding_periods:
            self.max_holding_period = max(holding_periods)
            self.avg_holding_period = sum(holding_periods) / len(holding_periods)
        else:
            self.max_holding_period = 0
            self.avg_holding_period = 0

    def calculate_dips(self):
        pass

    def calculate_benchmark(self):
        self.benchmark = (self.hist['close'][len(self.hist) - 1]/self.hist['open'][0])*self.capital
    
    def calculate_ratios(self):
        #for sharpe ratio, find return of portfolio and subtract risk free rate, then divide by standard deviation of portfolio
        #for sortino ratio, find return of portfolio and subtract risk free rate, then divide by standard deviation of negative returns
        
        static_balance_temp = pd.DataFrame(self.static_balance, columns=["balance"])
    
        # Calculate Sharpe ratio
        returns = static_balance_temp["balance"].pct_change()
        avg_return = returns.mean()
        risk_free_rate = 0.02
        std_dev = returns.std()
        self.sharpe_ratio = (avg_return - risk_free_rate) / std_dev

        # Calculate Sortino ratio
        downside_returns = returns[returns < 0]
        avg_downside_return = downside_returns.mean()
        downside_std_dev = downside_returns.std()
        self.sortino_ratio = (avg_downside_return - risk_free_rate) / downside_std_dev


        
    def calculate_metrics_static(self):
        # Calculate static performance metrics
        self.total_trades_static = self.trades
        self.win_trades_static = sum(pnl > 0 for pnl in self.pnl)
        self.lose_trades_static = self.total_trades_static - self.win_trades_static
        self.win_rate_static = self.win_trades_static / self.total_trades_static if self.total_trades_static > 0 else 0
        self.calculate_gross_profit()
        self.net_profit_static = self.total_pnl
        self.avg_profit_static = self.net_profit_static / (self.trades)
        self.largest_win = max(self.pnl)
        self.largest_loss = min(self.pnl)
        self.avg_win_static = (sum(pnl for pnl in self.pnl if pnl > 0)//self.win_trades_static) if self.win_trades_static > 0 else 0
        self.avg_loss_static = (sum(pnl for pnl in self.pnl if pnl < 0)//self.lose_trades_static) if self.lose_trades_static > 0 else 0
        self.calculate_max_drawdown()
        self.calculate_holding_periods()
        self.calculate_dips()
        self.calculate_benchmark()
        self.calculate_ratios()

    def calculate_metrics_compound(self):
        # Calculate compound performance metrics
        self.initial_balance_compound = self.capital
        self.num_trades_compound = self.trades
        self.max_pnl_compound = max(self.pnl) 
        self.min_pnl_compound = min(self.pnl)
        self.peak_portfolio_balance_compound = max(self.compound_balance) if self.compound_balance else self.initial_balance_compound
        self.lowest_portfolio_balance_compound = min(self.compound_balance) 
        self.end_portfolio_balance_compound = self.compound_balance[-1] if self.compound_balance else self.initial_balance_compound

    
    def backtest(self, static):
        self.is_static = static
        for index, signal in enumerate(self.signals):
            if signal == 1 and self.is_closed:
                self.open_long(index)
            elif signal == -1 and not self.is_closed:
                self.close_long(index)
            elif signal == -1 and self.is_closed:
                self.open_short(index)
            elif signal == 1 and not self.is_closed:
                self.close_short(index)
            else:
                self.do_nothing(index)
        if not self.is_closed:
            if self.is_long:
                self.close_long(len(self.signals)-1)
            else:
                self.close_short(len(self.signals)-1)
        if not self.is_static:
            self.plot_graph()
            self.calculate_metrics_compound()
        else:    
            self.calculate_metrics_static()
        

In [12]:
# Create a backtest object
bt = backtest(hist, strat)

# Run the backtest in static mode
bt.backtest(static=True)

# Print the calculated metrics
print("Total Trades (Static):", bt.total_trades_static)
print("Winning Trades (Static):", bt.win_trades_static)
print("Losing Trades (Static):", bt.lose_trades_static)
print("Win Rate (Static):", bt.win_rate_static)
print("Gross Profit (Static):", bt.gross_profit)
print("Net Profit (Static):", bt.net_profit_static)
print("Average Profit (Static):", bt.avg_profit_static)
print("Largest Win (Static):", bt.largest_win)
print("Largest Loss (Static):", bt.largest_loss)
print("Average Win (Static):", bt.avg_win_static)
print("Average Loss (Static):", bt.avg_loss_static)
print("Max Drawdown (Static):", bt.max_drawdown)
print("Max Holding Period (Static):", bt.max_holding_period)
print("Average Holding Period (Static):", bt.avg_holding_period)
print("Max Dip (Static):", bt.max_dip)
print("Average Dip (Static):", bt.avg_dip)
print("Sharpe Ratio (Static):", bt.sharpe_ratio)
print("Sortino Ratio (Static):", bt.sortino_ratio)
print("Benchmark:", bt.benchmark)






Total Trades (Static): 9849
Winning Trades (Static): 5599
Losing Trades (Static): 4250
Win Rate (Static): 0.5684841100619352
Gross Profit (Static): 36737.68787482726
Net Profit (Static): 11798.519510058206
Average Profit (Static): 1.1979408579610322
Largest Win (Static): 208.6206404222887
Largest Loss (Static): -123.1383774271234
Average Win (Static): 6.0
Average Loss (Static): -6.0
Max Drawdown (Static): 0.0
Max Holding Period (Static): 36.0
Average Holding Period (Static): 1.7884049142044878
Max Dip (Static): 0
Average Dip (Static): 0
Sharpe Ratio (Static): -0.510665725980229
Sortino Ratio (Static): -0.4841076138197998
Benchmark: 3111.40922960268
